## Homework 3: Model size reduction

__Soft deadline 26.10.25 23:59__   
__Hard deadline 28.10.25 23:59__

### About the assignment

In this assignment, you'll learn to solve the Named Entity Recognition (NER) problem on the most popular dataset, [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). You'll be given a pretrained BERT, which you'll need to downsize to a 20M parameter size with minimal loss in quality. To do this, you'll implement embedding factorization, distillation, parameter sharing, and so on.

This assignment will require you to conduct quite a few experiments, so we recommend not writing all the code in a notebook, but creating separate files for individual logical blocks and compiling everything into a project. This will keep your notebook small and make the task much easier for both you and your reviewers. Also, try to log all your experiments in wandb to ensure nothing gets lost.

### Grading and Penalties

The maximum grade for this assignment is __10 points__.

The grade for this homework assignment will be calculated based on the grade for the __assignment solution__ and the __report__, in which you are required to write about your work. You can earn up to 2 points for the report, but if you do not submit a report, no points will be awarded for the corresponding tasks. The tasks are divided into two parts: _mandatory_ and _for choice_. _mandatory_ tasks can earn a total of 6 points, while _for choice_ tasks can earn up to 14. This means you can earn 22 points for all homework (but don't get too excited, it's not that easy). Anything over 10 will be considered a bonus.

This assignment is to be completed independently. "Similar" solutions are considered plagiarism, and all students involved (including those from whom the plagiarism was committed) cannot receive more than 0 points for it. All code must be written independently. Using someone else's code is prohibited, even with a link to the source. Within reasonable limits, of course. Taking a couple of obvious lines of code to implement a small feature is acceptable.

Ineffective code implementation may negatively impact your grade. Your grade may also be reduced for poorly written code and poorly formatted plots. All answers must be accompanied by the code or comments explaining how it was derived.

### About the dataset

Named Entity Recognition is a task of classifying tokens into entity classes. CoNLL-2003 uses the **BIO** (Beggining, Inside, Outside) tagging system to name entities. Tags mean the following:

- *B-{tag}* – Beginning of an entity *{tag}*
- *I-{tag}* – Continuation of an entity *{tag}*
- *O* – Not an entity

Other tagging methods also exist, such as BILUO. You can read about them [here](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)) and [here](https://www.youtube.com/watch?v=dQw4w9WgXcQ).

There are a total of 9 different tags in the dataset.
- O – No entities correspond to the word. - B-PER/I-PER – a word or set of words corresponds to a specific _person_.
- B-ORG/I-ORG – a word or set of words corresponds to a specific _organization_.
- B-LOC/I-LOC – a word or set of words corresponds to a specific _location_.
- B-MISC/I-MISC – a word or set of words corresponds to an entity that does not belong to any of the previous categories. For example, a nationality, a work of art, an event, etc.

Let's get started!

We'll start with loading and preprocessing the dataset.

In [6]:
from datasets import load_dataset

dataset = load_dataset("eriktks/conll2003")

dataset = dataset.remove_columns(["id", "pos_tags", "chunk_tags"])
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

In [3]:
dataset['train'][0]

{'words': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [8]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [10]:
words = dataset["train"][0]["tokens"]
labels = dataset["train"][0]["ner_tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


### Preprocessing

Throughout the homework, we'll use the _cased_ version of BERT, meaning the tokenizer will be case-sensitive. For the NER task, case is important because names, organizations, and art objects are often capitalized, and it would be foolish to hide such information from the model.

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

During tokenization, words may be split into multiple tokens (like the word 'Fischler' in the example below), resulting in a discrepancy between the number of tokens and the number of labels. We'll have to resolve this discrepancy manually.

In [3]:
example = dataset["train"][12]
words = example["tokens"]
tags = [label_names[t] for t in example["ner_tags"]]
tokenized_text = tokenizer(example["tokens"], is_split_into_words=True)

print('Words: ', words)
print('Tokens:', tokenized_text.tokens())
print('Tags:', tags)

Words:  ['Only', 'France', 'and', 'Britain', 'backed', 'Fischler', "'s", 'proposal', '.']
Tokens: ['[CLS]', 'Only', 'France', 'and', 'Britain', 'backed', 'Fi', '##sch', '##ler', "'", 's', 'proposal', '.', '[SEP]']
Tags: ['O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'O', 'O', 'O']


__Task 1 (1 point).__ Tokenize the entire dataset and, for each text, align the tokens with the labels so that each token corresponds to one label. It's important to preserve the BIO notation. And don't forget about the special tokens! The result should look something like this:

In [1]:
aligned_labels = align_labels_with_tokens(example["ner_tags"], tokenized_text)
tags = [label_names[t] if t > -1 else t for t in aligned_labels]
print("Aligned labels:", aligned_labels)
print("Aligned label names:", tags)

Aligned labels: [-100    0    5    0    5    0    1    2    2    0    0    0    0 -100]
Aligned label names: [-100, 'O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', -100]


In [ ]:
# your code here

### Metric

The F1 score with micro-averaging is commonly used to evaluate NER quality. We'll load it from the `seqeval` library. The `f1_score` function accepts two 2D lists with correct and predicted labels, written as text, and returns the F1 score for them. You can use it with the default parameters.

In [12]:
# ! pip install seqeval

In [46]:
from seqeval.metrics import f1_score

A peculiarity of the F1 score for NER is that in some situations, incorrect answers can be counted as correct. For example, if the model predicted ['I-PER', 'I-PER'] , we can guess that the actual answer should be ['B-PER', 'I-PER'] , since an entity cannot begin with 'I-' . The `f1_score` function takes this into account and therefore only works with textual representations of labels.

### Model

We'll use `bert-base-cased` as the base model. As you can imagine, it wasn't trained on the NER task. Therefore, before we can reduce the size of BERT, we need to fine-tune it.

__Task 2 (1 point)__ Fine-tune `bert-base-cased` on our dataset using standard fine-tuning. You should achieve at least 0.9 F1 on the test set. Note that the higher the quality of the large model, the better the distilled learner will perform. You can use the `Trainer` from Hugging Face for training.

In [2]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', num_labels=len(label_names))

print('Number of parameters:', sum(p.numel() for p in model.parameters()))

Number of parameters: 107726601


In [44]:
# your code here

### Factorization of the embedding matrix

You can see that the embedding matrix currently takes up $V \cdot H = 28996 \cdot 768 = 22,268,928$ parameters. That's a fifth part of the entire model! Let's try to do something about it. The [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) model proposes factoring the embedding matrix into the product of two smaller matrices. Thus, the embedding parameters will contain $V \cdot E + E \cdot H$ elements, which is much smaller than $V \cdot H$ if $H \gg E$. The authors choose $E = 128$, but nothing prevents us from choosing any other value. For example, by choosing $H = 64$, we reduce the number of parameters by approximately 20M.

__Task 3 (1 point).__ Write a wrapper class over the embedding layer that implements factorization into two matrices, and fine-tune the factored model. Note that both matrices can be initialized using the SVD factorization to ensure a good initial approximation. This will save a significant amount of time on fine-tuning. With a factorization rank of $H = 64$, you should get an F1 score greater than 0.87.

In [ ]:
# your code here

### Knowledge Distillation

Knowledge distillation is a training paradigm in which the knowledge of a teacher model is distilled into a student model. The student can be any smaller model solving the same problem, but typically the student has the same architecture as the teacher. Distillation utilizes two loss functions:

1. Standard cross-entropy.
1. A function defining the distance between the prediction distributions of the teacher and student. Most commonly KL divergence is used.

To prevent the teacher's prediction distribution from becoming degenerate, a temperature greater than 1, such as 2 or 5, is added to the softmax.
__Important:__ when dividing the logits by the temperature, the gradient values ​​are reduced by a factor of $\tau^2$ (check this!). Therefore, to return them to their original scale, the error must be multiplied by $\tau^2$. More details can be found in Section 2.1 [of the original paper](https://arxiv.org/pdf/1503.02531).

<img src="https://intellabs.github.io/distiller/imgs/knowledge_distillation.png" width="800">

__Task 4 (3 points).__ Implement the knowledge distillation method shown in the image. To calculate the error between the student and teacher predictions, use the KL divergence [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (pay attention to the format of its inputs). To obtain the final error, sum the soft loss (2) with the hard loss (1).

Take the fine-tuned BERT from task 2 as a teacher and an untrained model with __no more than 20M__ parameters as a student. You can use embedding matrix factorization to reduce the number of parameters. If you've done everything correctly, you should achieve an F1 value of at least 0.7 on the test set. This should take about 20,000 training iterations. If you're having trouble, you can refer to the [DistillBERT](https://arxiv.org/abs/1910.01108) and [this article](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT).


__Important:__
* Don't forget to add _warmup_ when training the student.
* Don't forget to put the teacher into _eval_ mode.

In [ ]:
# your code here

# Tasks to choose from

As you can imagine, there are many more ways to downsize a trained model. In this section, you'll be asked to implement various methods to choose from. Each method carries a different number of points, depending on its complexity. Successful implementation will be assessed both by the code and by the quality of the test set. Any points you earn for this assignment beyond 10 will be considered bonus points.

In task 4, you trained a model with a parameter limit of __20M__. When implementing the methods in this section, adhere to the same limitation. This will allow you to fairly compare methods and draw the right conclusions. Please include everything you've tried in your report.

* __Parameter sharing (1 point).__ The BERT modification [ALBERT](https://arxiv.org/pdf/1909.11942.pdf), in addition to factoring embeddings, proposes sharing weights between layers. That is, different layers use the same weights. This technique is equivalent to using the same layer multiple times. It allows for a significant reduction in the number of parameters without losing much quality.
* __Factorization of intermediate layers (1 point).__ If the embedding matrix can be factorized, then everything else can be too. There are many different approaches to factorizing layers, and choosing just one is difficult. You can be inspired by [this list](https://lechnowak.com/posts/neural-network-low-rank-factorization-techniques/), find something else online, or come up with your own method. In any case, please justify your solution in your report.
* __Approximating intermediate layers (2 points).__ We discussed that in addition to approximating the student model outputs to the teacher model outputs, it is possible to approximate the outputs of intermediate layers. [This paper](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT) describes in detail how this can be done.
* __Pruning (4 points).__ The [SparseGPT](https://arxiv.org/abs/2301.00774) method proposes an approach that removes model weights once after training. It turns out that it is possible to remove up to half of all weights without losing quality. The mathematics behind the technique is quite complex, but the general approach is simple: we remove weights in each layer individually, and when we remove some of the weights in a layer, the remaining weights are reconfigured so that the overall output of the layer does not change.
* __Head Removal (6 points).__ Currently, we preserve all attention heads, but several studies show that most can be discarded without loss of quality. This [paper](https://arxiv.org/pdf/1905.09418.pdf) proposes an approach that adds gates to the attention mechanism, which regulate which heads participate in a layer and which do not. During training, the gates are adjusted so that most heads are not used. At the end of training, unused heads can be removed. This task is worth a lot of points because the method has fairly complex mathematics and the approach is difficult to implement. If you decide to invest your time in this and fail, we still will award you with some points based on the report.
__Tip:__ During training, carefully monitor the behavior of the gates. If you did everything correctly, they should be converge to zero. However, they do not always converge immediately; you need to give them time and train the model for a while.